# 🧠 K-Fold Cross-Validation from Scratch

Welcome to the hands-on explanation notebook for **Cross-Validation**! In this notebook, we will:
1. Explain the mechanics of $K$-Fold Cross-Validation.
2. Implement a **K-Fold splitter from scratch** using NumPy index partitioning.
3. Train a simple classifier (e.g. `DecisionTreeClassifier`) using $K$-Fold validation to compute fold accuracies, mean, and standard deviation.
4. Verify our scratch implementation splits against `scikit-learn`'s `KFold`.
5. Discuss **Stratified K-Fold** for imbalanced datasets.
6. Outline how $K$-Fold is used in deep learning and YOLO ensembling.

Let's start by importing the necessary libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import KFold

# Set seed for reproducibility
np.random.seed(42)

## 1. Simulating a Dataset

We generate a simple 2D dataset with 150 samples and 2 classes.

In [ ]:
# Generate synthetic dataset
X = np.random.rand(150, 2)
y = (X[:, 0] > 0.5).astype(int)
noise_idx = np.random.choice(150, 15, replace=False)
y[noise_idx] = 1 - y[noise_idx]

## 2. Implementing K-Fold Cross-Validation from Scratch

Let's write a `CustomKFold` splitter class:
-   `__init__(self, n_splits=5, shuffle=True, random_state=None)`
-   `split(self, X)`:
    -   Shuffle index array.
    -   Divide indices into `n_splits` arrays (folds).
    -   Yield `(train_idx, val_idx)` for each fold.

In [ ]:
class CustomKFold:
    def __init__(self, n_splits=5, shuffle=True, random_state=None):
        self.n_splits = n_splits
        self.shuffle = shuffle
        self.random_state = random_state

    def split(self, X):
        m = X.shape[0]
        indices = np.arange(m)
        
        if self.shuffle:
            if self.random_state is not None:
                np.random.seed(self.random_state)
            np.random.shuffle(indices)
            
        # Determine sizes of folds
        fold_sizes = np.full(self.n_splits, m // self.n_splits)
        fold_sizes[:m % self.n_splits] += 1
        
        current = 0
        folds = []
        for size in fold_sizes:
            folds.append(indices[current:current + size])
            current += size
            
        for i in range(self.n_splits):
            val_idx = folds[i]
            train_idx = np.concatenate([folds[j] for j in range(self.n_splits) if j != i])
            yield train_idx, val_idx

# Verify splitter fold sizes
custom_kf = CustomKFold(n_splits=5, shuffle=True, random_state=42)
for idx, (train, val) in enumerate(custom_kf.split(X)):
    print(f"Fold {idx+1} | Train Size: {len(train)} | Val Size: {len(val)}")

## 3. Training and Evaluation Loop

Let's run a $K$-fold validation loop:
-   In each fold, train a `DecisionTreeClassifier` on the fold's training indices.
-   Evaluate accuracy on the fold's validation indices.
-   Calculate mean and standard deviation of fold accuracies.

In [ ]:
fold_accuracies = []

for idx, (train_idx, val_idx) in enumerate(custom_kf.split(X)):
    X_train, y_train = X[train_idx], y[train_idx]
    X_val, y_val = X[val_idx], y[val_idx]
    
    clf = DecisionTreeClassifier(max_depth=3, random_state=42)
    clf.fit(X_train, y_train)
    
    preds = clf.predict(X_val)
    acc = accuracy_score(y_val, preds)
    fold_accuracies.append(acc)
    print(f"Fold {idx+1} Accuracy: {acc * 100:.2f}%")

mean_acc = np.mean(fold_accuracies)
std_acc = np.std(fold_accuracies)

print(f"\nOverall K-Fold CV Accuracy: {mean_acc * 100:.2f}% (± {std_acc * 100:.2f}%)")

## 4. Visualizing Fold Accuracies

Let's visualize the stability of our model predictions across the folds.

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(range(1, 6), [acc * 100 for acc in fold_accuracies], color='skyblue', edgecolor='black')
plt.axhline(mean_acc * 100, color='red', linestyle='--', label=f'Mean Accuracy: {mean_acc*100:.2f}%')
plt.xlabel('Fold Number')
plt.ylabel('Validation Accuracy (%)')
plt.title('K-Fold Cross Validation Accuracies')
plt.ylim(0, 100)
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

## 💡 Stratified K-Fold (preserving class balance)
If Class 1 represented only $2\%$ of the dataset, simple random K-Fold might create a validation fold with 0 samples of Class 1.
To solve this, **Stratified K-Fold** splits positive and negative samples independently into K equal groups and then merges them back, ensuring that every fold retains the exact $2\%$ proportion of Class 1.

## 💡 Connection to Deep Learning & YOLO
In modern deep learning, training a single model (like YOLO on COCO) can take days. Doing 5-Fold cross-validation would multiply this cost by 5, which is often infeasible.
However, K-Fold is extremely popular in the following computer vision scenarios:
1.  **Small Custom Datasets:** If you have only 200 custom images, K-Fold prevents bias from a single train/val split.
2.  **Ensemble Inference (K-Fold Blending):** You train 5 models (one on each fold). At test time, you run the input image through all 5 models, and average their bounding box predictions (using Non-Maximum Suppression). This yields significantly higher robustness and precision.